## Notebook Set up

In [0]:
import mlflow
import mlflow.pyfunc

import pandas as pd
import numpy as np

from pyspark.sql import functions as F
from pyspark.sql.window import Window

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

In [0]:
## Load Pertinent Tables

In [0]:
facilities = spark.table("virtue_foundation_dataset.gold.facilities")
core = spark.table("virtue_foundation_dataset.gold.facilities_core")

pincode = spark.table("virtue_foundation_dataset.silver.india_post_pincode_directory_cleaned")

nfhs = spark.table("virtue_foundation_dataset.silver.nfhs_5_district_health_indicators_cleaned")


## Prepare Hospital Data

In [0]:
from pyspark.sql import functions as F

hospitals = facilities.join(core, "unique_id") \
    .select(
        "unique_id",
        "latitude",
        "longitude",
        "capacity",
        "numberDoctors"
    ) \
    .withColumn(
        "capacity_int",
        F.when(F.col("capacity").isin("null", "", "NA"), None)
         .otherwise(F.col("capacity"))
         .cast("int")
    ) \
    .withColumn(
        "doctors_int",
        F.when(F.col("numberDoctors").isin("null", "", "NA"), None)
         .otherwise(F.col("numberDoctors"))
         .cast("int")
    ) \
    .fillna({"capacity_int": 0, "doctors_int": 0}) \
    .dropna(subset=["latitude", "longitude"])


## Map Hospital → District (via nearest pincode)

In [0]:
def haversine_expr(lat1, lon1, lat2, lon2):
    return 2 * 6371 * F.asin(
        F.sqrt(
            F.pow(F.sin(F.radians(lat2 - lat1) / 2), 2) +
            F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) *
            F.pow(F.sin(F.radians(lon2 - lon1) / 2), 2)
        )
    )

pincode_small = pincode.select(
    F.col("latitude").alias("p_lat"),
    F.col("longitude").alias("p_lon"),
    "district"
)

joined = hospitals.crossJoin(pincode_small)

joined = joined.withColumn(
    "distance_to_pincode",
    haversine_expr(
        F.col("latitude"),
        F.col("longitude"),
        F.col("p_lat"),
        F.col("p_lon")
    )
)

window = Window.partitionBy("unique_id").orderBy("distance_to_pincode")

hospital_with_district = joined.withColumn("rank", F.row_number().over(window)) \
    .filter("rank = 1") \
    .drop("rank", "distance_to_pincode", "p_lat", "p_lon")

## Join To NFHS Dataset

In [0]:
df = hospital_with_district.join(
    nfhs,
    hospital_with_district["district"] == nfhs["district_name"],
    "left"
)

df = df.withColumn(
    "young_population",
    F.coalesce(F.col("population_below_age_15_percent"), F.lit(0))
)

df = df.withColumn(
    "adult_population",
    F.coalesce(F.col("women_age_15_to_49_interviewed"), F.lit(0)) +
    F.coalesce(F.col("men_age_15_to_54_interviewed"), F.lit(0))
)

df = df.withColumn(
    "older_population_proxy",
    (
        F.coalesce(F.col("women_age_15_plus_with_high_blood_pressure_percent"), F.lit(0)) +
        F.coalesce(F.col("men_age_15_plus_with_high_blood_pressure_percent"), F.lit(0)) +
        F.coalesce(F.col("women_age_15_plus_with_high_or_very_high_blood_sugar_percent"), F.lit(0)) +
        F.coalesce(F.col("men_age_15_plus_with_high_or_very_high_blood_sugar_percent"), F.lit(0))
    )
)

df = df.withColumn(
    "female_health_risk",
    (
        F.coalesce(F.col("all_women_age_15_to_49_anemic_percent"), F.lit(0)) +
        F.coalesce(F.col("women_age_15_plus_with_high_blood_pressure_percent"), F.lit(0))
    )
)

df = df.withColumn(
    "male_health_risk",
    (
        F.coalesce(F.col("men_age_15_plus_with_high_blood_pressure_percent"), F.lit(0)) +
        F.coalesce(F.col("men_age_15_plus_using_tobacco_percent"), F.lit(0))
    )
)

## Training Dataset

In [0]:
pdf = df.select(
    "unique_id",
    "latitude",
    "longitude",
    "capacity_int",
    "doctors_int",
    "young_population",
    "adult_population",
    "older_population_proxy",
    "female_health_risk",
    "male_health_risk"
).toPandas()

## Synthetic User Data For India

In [0]:
import numpy as np

np.random.seed(42)

n = len(pdf)

# -----------------------------
# ✅ AGE DISTRIBUTION (India-based)
# -----------------------------

age_groups = np.random.choice(
    ["child", "adult", "elder"],
    size=n,
    p=[0.26, 0.67, 0.07]   # India distribution
)

def sample_age(group):
    if group == "child":
        return np.random.randint(1, 15)
    elif group == "adult":
        # centered around median (~29)
        return int(np.clip(np.random.normal(30, 12), 15, 64))
    else:
        return np.random.randint(65, 90)

pdf["user_age"] = [sample_age(g) for g in age_groups]


# -----------------------------
# ✅ GENDER DISTRIBUTION
# -----------------------------

pdf["user_gender"] = np.random.choice(
    ["male", "female"],
    size=n,
    p=[0.515, 0.485]   # India ratio
)


# -----------------------------
# ✅ LOCATION SAMPLING
# -----------------------------
# Slightly biased around hospital density, but realistic variation

pdf["user_lat"] = pdf["latitude"] + np.random.normal(0, 0.05, n)
pdf["user_lon"] = pdf["longitude"] + np.random.normal(0, 0.05, n)


Distance Calculation

In [0]:
def haversine_np(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

pdf["distance_km"] = haversine_np(
    pdf["user_lat"],
    pdf["user_lon"],
    pdf["latitude"],
    pdf["longitude"]
)

## AGE + GENDER ALIGNMENT

In [0]:
def age_gender_alignment(row):

    # Age component
    if row["user_age"] < 15:
        age_score = row["young_population"]
    elif row["user_age"] > 60:
        age_score = row["older_population_proxy"]
    else:
        age_score = row["adult_population"]

    # Gender component
    if row["user_gender"] == "female":
        gender_score = row["female_health_risk"]
    else:
        gender_score = row["male_health_risk"]

    return age_score + gender_score

pdf["alignment_score"] = pdf.apply(age_gender_alignment, axis=1)

## ML TRAINING

In [0]:
pdf["distance_score"] = np.exp(-pdf["distance_km"] / 20)
pdf["capacity_score"] = np.log1p(pdf["capacity_int"])
pdf["doctor_score"] = np.log1p(pdf["doctors_int"])


pdf["label"] = (
    0.6 * pdf["distance_score"] +     # ✅ dominant signal
    0.15 * pdf["capacity_score"] +
    0.15 * pdf["doctor_score"] +
    0.1 * pdf["alignment_score"]
)


In [0]:
X = pdf[[
    "distance_km",
    "capacity_int",
    "doctors_int",
    "user_age",
    "alignment_score"
]]

y = pdf["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y)

model = RandomForestRegressor   m nhua(n_estimators=100)
model.fit(X_train, y_train)

class HospitalModel(mlflow.pyfunc.PythonModel):

    def __init__(self, hospital_df, model):
        self.hospitals = hospital_df
        self.model = model

    def haversine(self, lat1, lon1, lat2, lon2):
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        
        a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
        return 6371 * 2 * np.arcsin(np.sqrt(a))

    def predict(self, context, model_input):

        user_age = model_input["user_age"].iloc[0]
        user_lat = model_input["user_lat"].iloc[0]
        user_lon = model_input["user_lon"].iloc[0]
        user_gender = model_input["user_gender"].iloc[0]

        df = self.hospitals.copy()

        # ✅ Step 1: Compute distance
        df["distance_km"] = self.haversine(
            user_lat, user_lon,
            df["latitude"], df["longitude"]
        )

        # ✅ 🚨 Step 2: FILTER (THIS IS WHAT YOU'RE MISSING)
        df = df[df["distance_km"] <= 100]

        if len(df) < 10:
            df = df.sort_values("distance_km").head(50)

        # ✅ Step 3: Feature engineering
        df["distance_score"] = np.exp(-df["distance_km"] / 20)

        df["capacity_score"] = np.log1p(df["capacity_int"])
        df["doctor_score"] = np.log1p(df["doctors_int"])

        # Age + gender logic (your existing code)
        if user_age < 15:
            age_score = df["young_population"]
        elif user_age > 60:
            age_score = df["older_population_proxy"]
        else:
            age_score = df["adult_population"]

        if user_gender == "female":
            gender_score = df["female_health_risk"]
        else:
            gender_score = df["male_health_risk"]

        df["alignment_score"] = age_score + gender_score
        df["user_age"] = user_age

        # ✅ Step 4: Predict
        features = df[[
            "distance_km",
            "capacity_int",
            "doctors_int",
            "user_age",
            "alignment_score"
        ]]

        df["score"] = self.model.predict(features)

        # ✅ Step 5: Return nearest-best hospitals
        return df.sort_values("score", ascending=False)[
            ["unique_id", "score", "distance_km"]
        ].head(10)


## LOG WITH ML FLOW

In [0]:
from mlflow.models.signature import infer_signature
import pandas as pd

# ✅ 1. Prepare serving dataset (already correct)
hospital_pd = pdf.drop_duplicates("unique_id")

# ✅ 2. Instantiate model
pyfunc_model = HospitalModel(hospital_pd, model)

# ✅ 3. Define example input (what your API will receive)
input_example = pd.DataFrame([{
    "user_age": 65,
    "user_gender": "female",
    "user_lat": 28.61,
    "user_lon": 77.20
}])

# ✅ 4. Generate example output
example_output = pyfunc_model.predict(None, input_example)

# ✅ 5. Infer signature
signature = infer_signature(input_example, example_output)

# ✅ 6. Log model with signature
with mlflow.start_run():

    mlflow.pyfunc.log_model(
        artifact_path="hospital_model",   # still fine in free edition
        python_model=pyfunc_model,
        signature=signature,
        input_example=input_example
    )

In [0]:

test_input = pd.DataFrame([{
    "user_age": 45,
    "user_gender": "male",
    "user_lat": 28.6139,
    "user_lon": 77.2090
}])

test_model = HospitalModel(hospital_pd, model)

result = test_model.predict(None, test_input)

display(result)
